In [1]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout[-3000:] if result.stdout else "")
    if result.stderr: print("STDERR:", result.stderr[-500:])

run("git clone https://github.com/slei109/PATNet.git /kaggle/working/PATNet")
run("git clone https://github.com/Lilacis/CD_TG.git /kaggle/working/CD_TG")
run("pip install open-clip-torch peft gdown -q")
print("Done!")


STDERR: Cloning into '/kaggle/working/PATNet'...


STDERR: Cloning into '/kaggle/working/CD_TG'...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00

Done!


In [2]:
import gdown, os

gdown.download_folder(
    'https://drive.google.com/drive/folders/1BICJ2kXZe0TtgB9PrUNJRfE3IaPF_fJR',
    output='/kaggle/working/clip_weights',
    quiet=False
)

print(os.listdir('/kaggle/working/clip_weights'))

Retrieving folder contents


Processing file 1VMDryounhIc-boJe4wRnFk24jNdg3gK6 RN50.pt
Processing file 1ep495t3VmPFlbrOTxZ8O7RjWzrUyNMom ViT-B-16.pt
Processing file 1tY28hHhNJ9UyqyVWATkedBkMdzzFPzsc ViT-B-32.pt


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1VMDryounhIc-boJe4wRnFk24jNdg3gK6
From (redirected): https://drive.google.com/uc?id=1VMDryounhIc-boJe4wRnFk24jNdg3gK6&confirm=t&uuid=a72d3ee8-8085-47ec-99c5-ada1098f33cf
To: /kaggle/working/clip_weights/RN50.pt
100%|██████████| 256M/256M [00:02<00:00, 94.5MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1ep495t3VmPFlbrOTxZ8O7RjWzrUyNMom
From (redirected): https://drive.google.com/uc?id=1ep495t3VmPFlbrOTxZ8O7RjWzrUyNMom&confirm=t&uuid=e07cdf2c-b256-483f-8d8b-5046ecb1b01f
To: /kaggle/working/clip_weights/ViT-B-16.pt
100%|██████████| 351M/351M [00:04<00:00, 85.6MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1tY28hHhNJ9UyqyVWATkedBkMdzzFPzsc
From (redirected): https://drive.google.com/uc?id=1tY28hHhNJ9UyqyVWATkedBkMdzzFPzsc&confirm=t&uuid=3648a573-6573-46f7-8996-a652f2a1147c
To: /k

['ViT-B-16.pt', 'ViT-B-32.pt', 'RN50.pt']



Download completed


In [3]:
# Fix patnet.py pretrained weights
patnet_path = '/kaggle/working/PATNet/model/patnet.py'
with open(patnet_path, 'r') as f:
    content = f.read()
content = content.replace(
    'resnet.resnet50(pretrained=True)',
    'resnet.resnet50(weights=resnet.ResNet50_Weights.IMAGENET1K_V1)'
)
with open(patnet_path, 'w') as f:
    f.write(content)
print("patnet.py fixed ✓")

# Fix encoder.py absolute path
encoder_path = '/kaggle/working/CD_TG/encoder.py'
with open(encoder_path, 'r') as f:
    content = f.read()
old = "category_file = category_file_map.get(model_name.lower(), 'clip_prompts/default.json')"
new = """_clip_dir = os.path.dirname(os.path.abspath(__file__))
        category_file = os.path.join(_clip_dir, category_file_map.get(model_name.lower(), 'clip_prompts/default.json'))"""
content = content.replace(old, new)
with open(encoder_path, 'w') as f:
    f.write(content)
print("encoder.py fixed ✓")

patnet.py fixed ✓
encoder.py fixed ✓


In [4]:
lung_path = '/kaggle/working/PATNet/data/lung.py'
with open(lung_path, 'r') as f:
    content = f.read()
content = content.replace(
    "os.path.join(datapath, 'LungSegmentation')",
    "os.path.join(datapath, 'Lung Segmentation')"
)
with open(lung_path, 'w') as f:
    f.write(content)

# Check actual path
import os
data_root = "/kaggle/input/chest-xray-masks-and-labels"
for root, dirs, files in os.walk(data_root):
    depth = root.replace(data_root,'').count(os.sep)
    if depth < 2:
        print(root)

In [6]:
code = r"""
from functools import reduce
from operator import add
import os, sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet

from model.base.feature import extract_feat_res
from model.base.correlation import Correlation
from model.learner import HPNLearner

CDTG_DIR = '/kaggle/working/CD_TG'
sys.path.insert(0, os.path.abspath(CDTG_DIR))
from module import ADaIN, AttentionModule, FFAttention
from encoder import CLIPModel


class PATNetCDTG(nn.Module):
    def __init__(self, backbone='resnet50', clip_model_name="ViT-B-16",
                 clip_model_path=None, alpha=0.3):
        super(PATNetCDTG, self).__init__()

        self.alpha    = alpha
        self.cdtg_dir = os.path.abspath(CDTG_DIR)

        # ── Backbone (ResNet50 only) ───────────────────────────────────────────
        self.backbone     = resnet.resnet50(weights=resnet.ResNet50_Weights.IMAGENET1K_V1)
        self.feat_ids     = list(range(4, 17))
        self.extract_feats = extract_feat_res
        nbottlenecks      = [3, 4, 6, 3]

        self.reference_layer3 = nn.Linear(2048, 2, bias=True)
        self.reference_layer2 = nn.Linear(1024, 2, bias=True)
        self.reference_layer1 = nn.Linear(512,  2, bias=True)
        for layer in [self.reference_layer1, self.reference_layer2, self.reference_layer3]:
            nn.init.kaiming_normal_(layer.weight, a=0, mode='fan_in', nonlinearity='linear')
            nn.init.constant_(layer.bias, 0)

        self.bottleneck_ids = reduce(add, list(map(lambda x: list(range(x)), nbottlenecks)))
        self.lids           = reduce(add, [[i+1]*x for i,x in enumerate(nbottlenecks)])
        self.stack_ids      = torch.tensor(self.lids).bincount().__reversed__().cumsum(dim=0)[:3]
        self.backbone.eval()
        self.hpn_learner    = HPNLearner(list(reversed(nbottlenecks[-3:])))
        self.cross_entropy_loss = nn.CrossEntropyLoss()

        # ── CD-TG modules ─────────────────────────────────────────────────────
        self.clip = CLIPModel(model_name=clip_model_name, model_path=clip_model_path)

        # Project 2048 -> 512 for attention (read-only, doesn't modify backbone feats)
        self.proj = nn.Conv2d(2048, 512, kernel_size=1)

        self.attention    = AttentionModule(embed_size=512)
        self.adain        = ADaIN(n_output=512, strength=0.3)
        self.ff_attention = FFAttention(weight_image=1-alpha, weight_text=alpha)

        # Project text-guided prototype back to 2048 for fusion
        self.proj_back = nn.Linear(512, 2048)

    def _get_text_embeddings(self, class_ids, benchmark, device):
        orig_dir = os.getcwd()
        os.chdir(self.cdtg_dir)
        text_emb = self.clip.forward(class_ids, benchmark)
        bg_ids   = torch.zeros(class_ids.shape[0], dtype=torch.long).to(device)
        text_emb_bg = self.clip.forward(bg_ids, 'background')
        os.chdir(orig_dir)
        return text_emb, text_emb_bg

    def forward(self, query_img, support_img, support_mask,
                class_ids=None, benchmark='isic', training=False):

        with torch.no_grad():
            query_feats   = self.extract_feats(query_img,   self.backbone,
                                               self.feat_ids, self.bottleneck_ids, self.lids)
            support_feats = self.extract_feats(support_img, self.backbone,
                                               self.feat_ids, self.bottleneck_ids, self.lids)
            support_feats_masked, prototypes_f, prototypes_b = self.mask_feature(
                support_feats, support_mask.clone())

        # ── CD-TG: enhance prototypes with text guidance ───────────────────────
        if class_ids is not None:
            text_emb, text_emb_bg = self._get_text_embeddings(class_ids, benchmark, query_img.device)

            n_feats     = len(support_feats_masked)   # 16
            stack_start = n_feats - 3                 # 13 -> feats 13,14,15 (all 2048ch)
            eps         = 1e-6

            for i in range(3):
                idx = stack_start + i

                # Project 2048->512 for text attention
                s_512 = self.proj(support_feats_masked[idx])   # [B, 512, H, W]

                if training:
                    s_512 = self.adain(s_512)

                # Text-guided support feature (SGU, paper eq. 2)
                s_text = self.attention(s_512, text_emb)        # [B, 512, H, W]

                # MAP: text-guided prototype (paper eq. 4 numerator)
                mask   = F.interpolate(support_mask.unsqueeze(1).float(),
                                       s_text.size()[2:], mode='bilinear', align_corners=True)
                proto_text = (s_text * mask).sum((2, 3)) / (mask.sum((2, 3)) + eps)  # [B, 512]

                # Project text proto back to 2048 to match baseline proto dim
                proto_text_2048 = self.proj_back(proto_text)    # [B, 2048]

                # Fuse: P_final = alpha * P_text + (1-alpha) * P_baseline  (paper eq. 4)
                prototypes_f[idx] = self.ff_attention(proto_text_2048, prototypes_f[idx])

        # ── PATNet Transformation + Correlation (unchanged, works on 2048ch) ──
        query_feats_t, support_feats_t = self.Transformation_Feature(
            query_feats, support_feats_masked, prototypes_f, prototypes_b)

        corr       = Correlation.multilayer_correlation(query_feats_t, support_feats_t, self.stack_ids)
        logit_mask = self.hpn_learner(corr)
        logit_mask = F.interpolate(logit_mask, support_img.size()[2:],
                                   mode='bilinear', align_corners=True)
        return logit_mask

    def mask_feature(self, features, support_mask):
        eps = 1e-6
        prototypes_f, prototypes_b, bg_features, mask_features = [], [], [], []
        for idx, feature in enumerate(features):
            mask    = F.interpolate(support_mask.unsqueeze(1).float(),
                                    feature.size()[2:], mode='bilinear', align_corners=True)
            bg_mask = 1 - mask
            bg_features.append(features[idx] * bg_mask)
            mask_features.append(features[idx] * mask)
            features[idx] = features[idx] * mask
            prototypes_f.append(features[idx].sum((2,3)) / (mask.sum((2,3)) + eps))
            prototypes_b.append(bg_features[idx].sum((2,3)) / (bg_mask.sum((2,3)) + eps))
        return mask_features, prototypes_f, prototypes_b

    def Transformation_Feature(self, query_feats, support_feats, prototypes_f, prototypes_b):
        transformed_query_feats, transformed_support_feats = [], []
        bsz = query_feats[0].shape[0]
        for idx, feature in enumerate(support_feats):
            C   = torch.cat((prototypes_b[idx].unsqueeze(1), prototypes_f[idx].unsqueeze(1)), dim=1)
            eps = 1e-6
            if idx <= 3:
                R = self.reference_layer1.weight.expand(C.shape)
            elif idx <= 9:
                R = self.reference_layer2.weight.expand(C.shape)
            else:
                R = self.reference_layer3.weight.expand(C.shape)
            R = R / (((R*R).sum(dim=2, keepdim=True)).sqrt() + eps)
            C = C / (((C*C).sum(dim=2, keepdim=True)).sqrt() + eps)
            P = torch.matmul(torch.pinverse(C), R).permute(0, 2, 1)
            init_size        = query_feats[idx].shape
            query_feats[idx] = query_feats[idx].view(bsz, C.size(2), -1)
            transformed_query_feats.append(torch.matmul(P, query_feats[idx]).view(init_size))
            init_size = feature.shape
            feature   = feature.view(bsz, C.size(2), -1)
            transformed_support_feats.append(torch.matmul(P, feature).view(init_size))
        return transformed_query_feats, transformed_support_feats

    def predict_mask_nshot(self, batch, nshot, benchmark='isic'):
        logit_mask_agg = 0
        for s_idx in range(nshot):
            logit_mask = self(
                batch['query_img'], batch['support_imgs'][:, s_idx],
                batch['support_masks'][:, s_idx],
                class_ids=batch['class_id'], benchmark=benchmark, training=False
            )
            logit_mask_agg += logit_mask.argmax(dim=1)
            if nshot == 1:
                return logit_mask_agg
        bsz       = logit_mask_agg.size(0)
        max_vote  = logit_mask_agg.view(bsz, -1).max(dim=1)[0]
        max_vote  = torch.stack([max_vote, torch.ones_like(max_vote).long()]).max(dim=0)[0].view(bsz, 1, 1)
        pred_mask = logit_mask_agg.float() / max_vote
        pred_mask[pred_mask < 0.5] = 0
        pred_mask[pred_mask >= 0.5] = 1
        return pred_mask

    def compute_objective(self, logit_mask, gt_mask):
        bsz = logit_mask.size(0)
        logit_mask = logit_mask.view(bsz, 2, -1)
        gt_mask = gt_mask.view(bsz, -1).long().clamp(0, 1)
        return self.cross_entropy_loss(logit_mask, gt_mask)

    def train_mode(self):
        self.train()
        self.backbone.eval()
"""
with open('/kaggle/working/PATNet/patnet_cdtg.py', 'w') as f:
    f.write(code)
print("patnet_cdtg.py created ✓")

patnet_cdtg.py created ✓


In [11]:
!pip install tensorboardX
import sys, torch, torch.nn as nn
sys.path.insert(0, '/kaggle/working/PATNet')

from model.patnet import PATNetwork
from common.evaluation import Evaluator
from data.dataset import FSSDataset
from common import utils
from common.logger import AverageMeter

device = torch.device('cuda')

model = PATNetwork(backbone='resnet50')
weights = torch.load('/kaggle/input/datasets/satvikagada/bestmodel/best_model.pt', map_location=device)
model.load_state_dict(weights, strict=False)
model = nn.DataParallel(model)
model.to(device).eval()
print("PATNet loaded ✓")
Evaluator.initialize()
FSSDataset.initialize(img_size=400, datapath='/kaggle/input/chest-xray-masks-and-labels')

PATNet loaded ✓


In [15]:
import os
data_root = "/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels"
for root, dirs, files in os.walk(data_root):
    depth = root.replace(data_root, '').count(os.sep)
    if depth <= 2:
        print(f"{root} → {len(files)} files")

/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels → 0 files
/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/data → 0 files
/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/data/Lung Segmentation → 2 files
/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/Lung Segmentation → 2 files
/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/Lung Segmentation/ClinicalReadings → 800 files
/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/Lung Segmentation/.ipynb_checkpoints → 1 files
/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/Lung Segmentation/CXR_png → 800 files
/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/Lung Segmentation/masks → 704 files
/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/Lung Segmentation/test → 96 files


In [16]:
lung_path = '/kaggle/working/PATNet/data/lung.py'
with open(lung_path, 'r') as f:
    content = f.read()

# Fix base path
content = content.replace(
    "os.path.join(datapath, 'LungSegmentation')",
    "os.path.join(datapath, 'Lung Segmentation')"
)

with open(lung_path, 'w') as f:
    f.write(content)
print("Fixed ✓")

# Now reinitialize with correct datapath
Evaluator.initialize()
FSSDataset.initialize(img_size=400, 
    datapath='/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels')

dl = FSSDataset.build_dataloader('lung', 20, 0, 0, 'test', 1)
print(f"Dataset size: {len(dl.dataset)}")

Fixed ✓
Dataset size: 600


In [20]:
fix_code = '''r""" Chest X-ray few-shot semantic segmentation dataset """
import os
import glob

from torch.utils.data import Dataset
import torch.nn.functional as F
import torch
import PIL.Image as Image
import numpy as np


class DatasetLung(Dataset):
    def __init__(self, datapath, fold, transform, split, shot, num=600):
        self.split = split
        self.benchmark = 'lung'
        self.shot = shot
        self.num = num

        self.base_path = os.path.join(datapath, 'Lung Segmentation')
        self.img_path = os.path.join(self.base_path, 'CXR_png')
        self.ann_path = os.path.join(self.base_path, 'masks')

        self.categories = [\'1\']
        self.class_ids = range(0, 1)
        self.img_metadata_classwise = self.build_img_metadata_classwise()
        self.transform = transform

    def __len__(self):
        return self.num

    def __getitem__(self, idx):
        query_name, support_names, class_sample = self.sample_episode(idx)
        query_img, query_mask, support_imgs, support_masks = self.load_frame(query_name, support_names)

        query_img = self.transform(query_img)
        query_mask = F.interpolate(query_mask.unsqueeze(0).unsqueeze(0).float(), query_img.size()[-2:], mode=\'nearest\').squeeze()

        support_imgs = torch.stack([self.transform(support_img) for support_img in support_imgs])
        support_masks_tmp = []
        for smask in support_masks:
            smask = F.interpolate(smask.unsqueeze(0).unsqueeze(0).float(), support_imgs.size()[-2:], mode=\'nearest\').squeeze()
            support_masks_tmp.append(smask)
        support_masks = torch.stack(support_masks_tmp)

        return {\'query_img\': query_img, \'query_mask\': query_mask,
                \'query_name\': query_name, \'support_imgs\': support_imgs,
                \'support_masks\': support_masks, \'support_names\': support_names,
                \'class_id\': torch.tensor(class_sample)}

    def mask_to_img_name(self, mask_path):
        """Convert mask filename to image filename."""
        basename = os.path.basename(mask_path)
        # Remove _mask.png suffix and add .png
        img_name = basename.replace(\'_mask.png\', \'.png\')
        return img_name

    def load_frame(self, query_name, support_names):
        query_mask = self.read_mask(query_name)
        support_masks = [self.read_mask(name) for name in support_names]

        query_img_name = self.mask_to_img_name(query_name)
        query_img = Image.open(os.path.join(self.img_path, query_img_name)).convert(\'RGB\')

        support_imgs = []
        for name in support_names:
            img_name = self.mask_to_img_name(name)
            support_imgs.append(Image.open(os.path.join(self.img_path, img_name)).convert(\'RGB\'))

        return query_img, query_mask, support_imgs, support_masks

    def read_mask(self, img_name):
        mask = torch.tensor(np.array(Image.open(img_name).convert(\'L\')))
        mask[mask < 128] = 0
        mask[mask >= 128] = 1
        return mask

    def sample_episode(self, idx):
        class_id = idx % len(self.class_ids)
        class_sample = self.categories[class_id]
        query_name = np.random.choice(self.img_metadata_classwise[class_sample], 1, replace=False)[0]
        support_names = []
        while True:
            support_name = np.random.choice(self.img_metadata_classwise[class_sample], 1, replace=False)[0]
            if query_name != support_name:
                support_names.append(support_name)
            if len(support_names) == self.shot:
                break
        return query_name, support_names, class_id

    def build_img_metadata_classwise(self):
        img_metadata_classwise = {\'1\': []}
        img_paths = sorted(glob.glob(os.path.join(self.ann_path, \'*_mask.png\')))
        # Only keep masks that have a corresponding image
        for img_path in img_paths:
            img_name = self.mask_to_img_name(img_path)
            if os.path.exists(os.path.join(self.img_path, img_name)):
                img_metadata_classwise[\'1\'].append(img_path)
        print(f"Lung dataset: {len(img_metadata_classwise[\'1\'])} matched pairs found")
        return img_metadata_classwise
'''

with open('/kaggle/working/PATNet/data/lung.py', 'w') as f:
    f.write(fix_code)
print("lung.py rewritten ✓")

# Reinitialize
from importlib import reload
import data.lung
reload(data.lung)

from data.dataset import FSSDataset
from common.evaluation import Evaluator
Evaluator.initialize()
FSSDataset.initialize(img_size=400,
    datapath='/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels')
dl = FSSDataset.build_dataloader('lung', 20, 0, 0, 'test', 1)
print(f"Dataloader ready: {len(dl)} batches")

lung.py rewritten ✓
Dataloader ready: 30 batches


In [23]:
import os, glob

img_dir  = '/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/Lung Segmentation/CXR_png'
mask_dir = '/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/Lung Segmentation/masks'

imgs  = set([f.replace('.png','') for f in os.listdir(img_dir)  if f.endswith('.png')])
masks = set([f.replace('_mask.png','') for f in os.listdir(mask_dir) if f.endswith('_mask.png')])

print(f"Images: {len(imgs)}")
print(f"Masks:  {len(masks)}")
print(f"Matched: {len(imgs & masks)}")
print(f"Masks with no image: {list(masks - imgs)[:5]}")

Images: 800
Masks:  566
Matched: 566
Masks with no image: []


In [24]:
import importlib, sys

# Remove cached modules
for mod in list(sys.modules.keys()):
    if 'data' in mod or 'lung' in mod:
        del sys.modules[mod]

# Reimport fresh
sys.path.insert(0, '/kaggle/working/PATNet')
from data.dataset import FSSDataset
from common.evaluation import Evaluator

Evaluator.initialize()
FSSDataset.initialize(img_size=400,
    datapath='/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels')
dl = FSSDataset.build_dataloader('lung', 20, 0, 0, 'test', 1)
print(f"Dataloader ready: {len(dl)} batches")

# Verify one sample loads correctly
batch = next(iter(dl))
print("query_img shape:", batch['query_img'].shape)
print("query_mask unique:", batch['query_mask'].unique())

Lung dataset: 566 matched pairs found
Dataloader ready: 30 batches
query_img shape: torch.Size([20, 3, 400, 400])
query_mask unique: tensor([0., 1.])


In [29]:
utils.fix_randseed(0)
avg = AverageMeter(dl.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl):
        batch = utils.to_cuda(batch)
        pred = model.module.predict_mask_nshot(batch, nshot=1)
        assert pred.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred.clone(), batch)
        avg.update(inter, union, batch['class_id'], None)
        if idx % 5 == 0:
            miou_tmp, _ = avg.compute_iou()
            print(f"Batch {idx+1}/{len(dl)}  mIoU: {miou_tmp:.2f}", flush=True)

miou1, fb1 = avg.compute_iou()
print(f"\nPATNet 1-shot → mIoU: {miou1:.2f}  FB-IoU: {fb1:.2f}")

Batch 1/30  mIoU: 60.42
Batch 6/30  mIoU: 62.64
Batch 11/30  mIoU: 62.23
Batch 16/30  mIoU: 61.71
Batch 21/30  mIoU: 61.48
Batch 26/30  mIoU: 61.27

PATNet 1-shot → mIoU: 61.33  FB-IoU: 71.47


In [31]:
utils.fix_randseed(0)
dl5 = FSSDataset.build_dataloader('lung', 20, 0, 0, 'test', 5)
avg5 = AverageMeter(dl5.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl5):
        batch = utils.to_cuda(batch)
        pred = model.module.predict_mask_nshot(batch, nshot=5)
        assert pred.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred.clone(), batch)
        avg5.update(inter, union, batch['class_id'], None)
        if idx % 5 == 0:
            miou_tmp, _ = avg5.compute_iou()
            print(f"Batch {idx+1}/{len(dl5)}  mIoU: {miou_tmp:.2f}", flush=True)

miou5, fb5 = avg5.compute_iou()
print(f"\nPATNet 5-shot → mIoU: {miou5:.2f}  FB-IoU: {fb5:.2f}")

Lung dataset: 566 matched pairs found
Batch 1/30  mIoU: 61.58
Batch 6/30  mIoU: 62.79
Batch 11/30  mIoU: 63.14
Batch 16/30  mIoU: 63.53
Batch 21/30  mIoU: 63.64
Batch 26/30  mIoU: 63.44

PATNet 5-shot → mIoU: 63.36  FB-IoU: 73.12


In [33]:
encoder_path = '/kaggle/working/CD_TG/encoder.py'
with open(encoder_path, 'r') as f:
    content = f.read()

content = content.replace(
    'checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=weights_only)',
    'checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)'
)

with open(encoder_path, 'w') as f:
    f.write(content)
print("encoder.py fixed ✓")

encoder.py fixed ✓


In [35]:
factory_path = '/usr/local/lib/python3.12/dist-packages/open_clip/factory.py'
with open(factory_path, 'r') as f:
    content = f.read()

content = content.replace(
    'checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=weights_only)',
    'checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)'
)

with open(factory_path, 'w') as f:
    f.write(content)
print("factory.py fixed ✓")

factory.py fixed ✓


In [39]:
import open_clip, os
print(open_clip.__file__)

/usr/local/lib/python3.12/dist-packages/open_clip/__init__.py


In [40]:
factory_path = '/usr/local/lib/python3.12/dist-packages/open_clip/factory.py'
with open(factory_path, 'r') as f:
    content = f.read()
print(len(content))

# Find all lines with weights_only
for i, line in enumerate(content.split('\n')):
    if 'weights_only' in line:
        print(f"Line {i}: {line}")

51729
Line 155:         weights_only=True,
Line 163:             checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
Line 184:         weights_only: bool = True,
Line 193:     state_dict = load_state_dict(checkpoint_path, device=device, weights_only=weights_only)
Line 270:         weights_only: bool = True,
Line 307:         weights_only: Use weights_only=True for torch.load (safer).
Line 513:             weights_only=weights_only,
Line 528:                     weights_only=weights_only
Line 557:                     weights_only=weights_only
Line 851:         weights_only: bool = True,
Line 892:         weights_only: Use weights_only=True for torch.load (safer).
Line 948:         weights_only=weights_only,
Line 983:         weights_only: bool = True,
Line 1019:         weights_only: Use weights_only=True for torch.load (safer).
Line 1074:         weights_only=weights_only,


In [41]:
factory_path = '/usr/local/lib/python3.12/dist-packages/open_clip/factory.py'
with open(factory_path, 'r') as f:
    content = f.read()

# Fix all default values
content = content.replace('weights_only=True,', 'weights_only=False,')
content = content.replace('weights_only: bool = True,', 'weights_only: bool = False,')

with open(factory_path, 'w') as f:
    f.write(content)

# Verify
for i, line in enumerate(content.split('\n')):
    if 'weights_only' in line:
        print(f"Line {i}: {line}")

Line 155:         weights_only=False,
Line 163:             checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
Line 184:         weights_only: bool = False,
Line 193:     state_dict = load_state_dict(checkpoint_path, device=device, weights_only=weights_only)
Line 270:         weights_only: bool = False,
Line 307:         weights_only: Use weights_only=True for torch.load (safer).
Line 513:             weights_only=weights_only,
Line 528:                     weights_only=weights_only
Line 557:                     weights_only=weights_only
Line 851:         weights_only: bool = False,
Line 892:         weights_only: Use weights_only=True for torch.load (safer).
Line 948:         weights_only=weights_only,
Line 983:         weights_only: bool = False,
Line 1019:         weights_only: Use weights_only=True for torch.load (safer).
Line 1074:         weights_only=weights_only,


In [42]:
import importlib
import open_clip
importlib.reload(open_clip.factory)
importlib.reload(open_clip)
print("open_clip reloaded ✓")

open_clip reloaded ✓


In [43]:
import glob
sys.path.insert(0, '/kaggle/working/CD_TG')
from patnet_cdtg import PATNetCDTG

clip_files = glob.glob('/kaggle/working/clip_weights/**/*.pt', recursive=True)
print("CLIP files found:", clip_files)
clip_path = clip_files[0]

cdtg = PATNetCDTG(backbone='resnet50', clip_model_name='ViT-B-16',
                   clip_model_path=clip_path, alpha=0.3)
state = torch.load('/kaggle/input/datasets/satvikagada/bestmodel/best_model.pt', map_location=device)
model_dict = cdtg.state_dict()
matched = {k:v for k,v in state.items() if k in model_dict and model_dict[k].shape==v.shape}
model_dict.update(matched)
cdtg.load_state_dict(model_dict)
cdtg = nn.DataParallel(cdtg)
cdtg.to(device).eval()
print(f"CD-TG loaded, matched {len(matched)}/{len(model_dict)} keys ✓")

CLIP files found: ['/kaggle/working/clip_weights/ViT-B-16.pt', '/kaggle/working/clip_weights/ViT-B-32.pt', '/kaggle/working/clip_weights/RN50.pt']
CD-TG loaded, matched 424/438 keys ✓


In [44]:
utils.fix_randseed(0)
dl_cdtg = FSSDataset.build_dataloader('lung', 20, 0, 0, 'test', 1)
avg_cdtg = AverageMeter(dl_cdtg.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl_cdtg):
        batch = utils.to_cuda(batch)
        pred = cdtg.module.predict_mask_nshot(batch, nshot=1, benchmark='lung')
        assert pred.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred.clone(), batch)
        avg_cdtg.update(inter, union, batch['class_id'], None)
        if idx % 5 == 0:
            miou_tmp, _ = avg_cdtg.compute_iou()
            print(f"Batch {idx+1}/{len(dl_cdtg)}  mIoU: {miou_tmp:.2f}", flush=True)

miou_c1, fb_c1 = avg_cdtg.compute_iou()
print(f"\nCD-TG 1-shot → mIoU: {miou_c1:.2f}  FB-IoU: {fb_c1:.2f}")

Lung dataset: 566 matched pairs found
Batch 1/30  mIoU: 58.86
Batch 6/30  mIoU: 60.79
Batch 11/30  mIoU: 60.36
Batch 16/30  mIoU: 60.02
Batch 21/30  mIoU: 59.81
Batch 26/30  mIoU: 59.52

CD-TG 1-shot → mIoU: 59.58  FB-IoU: 69.86


In [45]:
utils.fix_randseed(0)
dl_cdtg5 = FSSDataset.build_dataloader('lung', 20, 0, 0, 'test', 5)
avg_cdtg5 = AverageMeter(dl_cdtg5.dataset)

with torch.no_grad():
    for idx, batch in enumerate(dl_cdtg5):
        batch = utils.to_cuda(batch)
        pred = cdtg.module.predict_mask_nshot(batch, nshot=5, benchmark='lung')
        assert pred.size() == batch['query_mask'].size()
        inter, union = Evaluator.classify_prediction(pred.clone(), batch)
        avg_cdtg5.update(inter, union, batch['class_id'], None)
        if idx % 5 == 0:
            miou_tmp, _ = avg_cdtg5.compute_iou()
            print(f"Batch {idx+1}/{len(dl_cdtg5)}  mIoU: {miou_tmp:.2f}", flush=True)

miou_c5, fb_c5 = avg_cdtg5.compute_iou()
print(f"\nCD-TG 5-shot → mIoU: {miou_c5:.2f}  FB-IoU: {fb_c5:.2f}")

Lung dataset: 566 matched pairs found
Batch 1/30  mIoU: 60.11
Batch 6/30  mIoU: 61.05
Batch 11/30  mIoU: 61.41
Batch 16/30  mIoU: 61.67
Batch 21/30  mIoU: 61.75
Batch 26/30  mIoU: 61.55

CD-TG 5-shot → mIoU: 61.47  FB-IoU: 71.43


In [46]:
print("\n" + "="*55)
print("  RESULTS — Chest X-Ray Lung Segmentation")
print("="*55)
print(f"  {'Method':<25} {'1-shot':>10} {'5-shot':>10}")
print(f"  {'-'*45}")
print(f"  {'PATNet (paper)':<25} {'66.6':>10} {'70.2':>10}")
print(f"  {'PATNet+CD-TG (paper)':<25} {'71.8':>10} {'74.3':>10}")
print(f"  {'-'*45}")
print(f"  {'PATNet (ours)':<25} {miou1.item():>10.2f} {miou5.item():>10.2f}")
print(f"  {'PATNet+CD-TG (ours)':<25} {miou_c1.item():>10.2f} {miou_c5.item():>10.2f}")
print("="*55)


  RESULTS — Chest X-Ray Lung Segmentation
  Method                        1-shot     5-shot
  ---------------------------------------------
  PATNet (paper)                  66.6       70.2
  PATNet+CD-TG (paper)            71.8       74.3
  ---------------------------------------------
  PATNet (ours)                  61.33      63.36
  PATNet+CD-TG (ours)            59.58      61.47


In [47]:
# Fair comparison — same episodes for both
utils.fix_randseed(0)
dl_fair = FSSDataset.build_dataloader('lung', 20, 0, 0, 'test', 1)

# PATNet
avg_pat = AverageMeter(dl_fair.dataset)
utils.fix_randseed(0)
with torch.no_grad():
    for idx, batch in enumerate(dl_fair):
        batch = utils.to_cuda(batch)
        pred = model.module.predict_mask_nshot(batch, nshot=1)
        inter, union = Evaluator.classify_prediction(pred.clone(), batch)
        avg_pat.update(inter, union, batch['class_id'], None)
miou_pat, _ = avg_pat.compute_iou()
print(f"PATNet 1-shot: {miou_pat:.2f}")

# CD-TG on same episodes
avg_cdtg = AverageMeter(dl_fair.dataset)
utils.fix_randseed(0)
with torch.no_grad():
    for idx, batch in enumerate(dl_fair):
        batch = utils.to_cuda(batch)
        pred = cdtg.module.predict_mask_nshot(batch, nshot=1, benchmark='lung')
        inter, union = Evaluator.classify_prediction(pred.clone(), batch)
        avg_cdtg.update(inter, union, batch['class_id'], None)
miou_cdtg, _ = avg_cdtg.compute_iou()
print(f"CD-TG 1-shot:  {miou_cdtg:.2f}")
print(f"Improvement:   {miou_cdtg - miou_pat:.2f}")

Lung dataset: 566 matched pairs found
PATNet 1-shot: 61.33
CD-TG 1-shot:  59.58
Improvement:   -1.75
